# JSON-LD demo — export, reload, reuse (interoperability)

Idea in one line: we **write** our data as JSON-LD files, and then **anyone** (here: a plain script with no Django and no API) can **read** them and combine them with other data.

1. **Export** — ask the project for JSON-LD and save it to `data/exports/`
2. **Look** at the file
3. **Reload** it from disk with a generic tool (`rdflib`) — no project code
4. **Use** it: questions with SPARQL
5. **Interoperability** — merge it with a file made by someone else

## 1 · Export (the only step that uses the project)

In [1]:
import sys, os, json
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "sources.yaml").exists())
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "webapp")]
os.environ["DJANGO_SETTINGS_MODULE"] = "threatgraph_site.settings"; os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
import django; django.setup()
from django.test import Client
from django.test.utils import setup_test_environment
from django.conf import settings
setup_test_environment(); settings.ALLOWED_HOSTS.append("127.0.0.1")
web = Client(HTTP_HOST="127.0.0.1:8000")                      # calls the API in-process: no server to start
LD = {"HTTP_ACCEPT": "application/ld+json"}                  # 'I want JSON-LD'

OUT = ROOT / "data" / "exports"; OUT.mkdir(parents=True, exist_ok=True)
def export(url, name):
    r = web.get(url, **LD); assert r.status_code == 200 and r["Content-Type"].startswith("application/ld+json"), url
    path = OUT / name; path.write_bytes(r.content); return path

f_cve  = export("/api/vulnerabilities/CVE-2024-34102/", "cve_2024_34102.jsonld")      # one CVE
f_tech = export("/api/techniques/T1190/",               "technique_T1190.jsonld")      # one technique (+ all its mappings)

# a small bundle: 10 critical CVEs mapped to T1190, details merged in one '@graph' file
ids = [v["cve_id"] for v in web.get("/api/vulnerabilities/", {"technique": "T1190", "severity": "CRITICAL", "page_size": 10}).json()["results"]]
docs = [web.get(f"/api/vulnerabilities/{c}/", **LD).json() for c in ids]
bundle = {"@context": docs[0]["@context"], "@graph": [{k: v for k, v in d.items() if k != "@context"} for d in docs]}
f_bundle = OUT / "critical_T1190_10cves.jsonld"; f_bundle.write_text(json.dumps(bundle, indent=1))
for f in (f_cve, f_tech, f_bundle): print(f"{f.name:32} {f.stat().st_size/1024:7.1f} KB")

cve_2024_34102.jsonld                8.0 KB
technique_T1190.jsonld              84.8 KB
critical_T1190_10cves.jsonld        54.7 KB


## 2 · Look at the file
It is normal JSON. Two things are new: **`@id`** (the public identifier of the thing — here the NVD page of the CVE) and **`@context`** (the dictionary that says what each key means).

In [6]:
doc = json.loads(f_cve.read_text())
print("keys of the context (first 8):", list(doc["@context"])[:8], "...")
print("cveId  ->", doc["@context"]["cveId"], "   (= Dublin Core 'identifier')")
print("title  ->", doc["@context"]["title"])
print()
print(json.dumps({k: doc[k] for k in ("@id", "@type", "cveId", "vendorProject", "product", "dateAdded")}, indent=1))
print("first mapping:", json.dumps(doc["mappings"][0], indent=1)[:700])

keys of the context (first 8): ['tg', 'dcterms', 'dcat', 'prov', 'rdfs', 'xsd', 'Vulnerability', 'NvdRecord'] ...
cveId  -> dcterms:identifier    (= Dublin Core 'identifier')
title  -> dcterms:title

{
 "@id": "https://nvd.nist.gov/vuln/detail/CVE-2024-34102",
 "@type": "Vulnerability",
 "cveId": "CVE-2024-34102",
 "vendorProject": "Adobe",
 "product": "Commerce and Magento Open Source",
 "dateAdded": "2024-07-17"
}
first mapping: {
 "@id": "urn:threatgraph:pkev-mapping:1",
 "@type": "Mapping",
 "mappingType": "exploitation_technique",
 "capabilityGroup": "xxe",
 "comments": "This vulnerability is exploited by sending a crafted XML document that references external entities with the likely goal of accessing local data.",
 "evidenceUrl": [
  "https://www.assetnote.io/resources/research/why-nested-deserialization-is-harmful-magento-xxe-cve-2024-34102",
  "https://www.vicarius.io/vsociety/posts/cosmicsting-critical-unauthenticated-xxe-vulnerability-in-adobe-commerce-and-magento-cve-2024-3

## 3 · Reload — from here on: no Django, no API, no project code
Only the three files and a generic library. This is what **another team** would do with our export.

In [7]:
from rdflib import Graph
g = Graph()
for f in (f_cve, f_tech, f_bundle):
    g.parse(f, format="json-ld")                       # read the file, expand the context, get triples
print(f"{len(g):,} triples from 3 files")
# the same information as 'triples' (Turtle): subject  predicate  object
one = Graph().parse(f_cve, format="json-ld"); one.bind("tg", "https://example.org/threatgraph/vocab#")
print(one.serialize(format="turtle")[:1500])

1,944 triples from 3 files
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix tg: <https://example.org/threatgraph/vocab#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://nvd.nist.gov/vuln/detail/CVE-2024-34102> a tg:Vulnerability ;
    dcterms:identifier "CVE-2024-34102" ;
    dcterms:title "Adobe Commerce and Magento Open Source Improper Restriction of XML External Entity Reference (XXE) Vulnerability" ;
    prov:wasDerivedFrom <https://github.com/cisagov/kev-data/tree/0cca62ed469c333a45760beeaf35fcc53352aedd> ;
    tg:apiUrl <http://127.0.0.1:8000/api/vulnerabilities/CVE-2024-34102/> ;
    tg:dateAdded "2024-07-17"^^xsd:date ;
    tg:dueBeforeAdded false ;
    tg:dueDate "2024-08-07"^^xsd:date ;
    tg:hasMapping <urn:threatgraph:pkev-mapping:1>,
        <urn:threatgraph:pkev-mapping:2>,
        <urn:threatgraph:pkev-mapping:3> ;
    tg:notes "https://helpx.adobe.c

## 4 · Use it — questions in SPARQL (a standard query language)

In [8]:
import pandas as pd
P = "PREFIX tg: <https://example.org/threatgraph/vocab#> PREFIX dct: <http://purl.org/dc/terms/> PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> "
def ask(sparql):
    res = g.query(P + sparql); return pd.DataFrame([[str(x) for x in row] for row in res], columns=[str(v) for v in res.vars])

print("Q1. Which ATT&CK techniques and tactics does CVE-2024-34102 involve?")
display(ask('''SELECT ?technique ?name ?tactic ?role WHERE {
  <https://nvd.nist.gov/vuln/detail/CVE-2024-34102> tg:hasMapping ?m .
  ?m tg:technique ?t ; tg:mappingType ?role .
  ?t tg:techniqueId ?technique ; rdfs:label ?name . OPTIONAL { ?t tg:tactic/tg:shortname ?tactic } }'''))
print("Q2. All critical CVEs loaded (10 from the bundle + CVE-2024-34102 from the single file): vendor, product, score")
display(ask('''SELECT ?cve ?vendor ?product ?score WHERE {
  ?v dct:identifier ?cve ; tg:vendorProject ?vendor ; tg:product ?product ; tg:nvdRecord/tg:cvssV31/tg:baseScore ?score } ORDER BY ?cve'''))

Q1. Which ATT&CK techniques and tactics does CVE-2024-34102 involve?


,technique,name,tactic,role
0,T1190,Exploit Public-Facing Application,initial-access,exploitation_technique
1,T1059,Command and Scripting Interpreter,execution,primary_impact
2,T1005,Data from Local System,collection,secondary_impact


Q2. All critical CVEs loaded (10 from the bundle + CVE-2024-34102 from the single file): vendor, product, score


,cve,vendor,product,score
0,CVE-2010-2861,Adobe,ColdFusion,9.8
1,CVE-2013-0625,Adobe,ColdFusion,9.8
2,CVE-2013-0632,Adobe,ColdFusion,9.8
3,CVE-2014-6271,GNU,Bourne-Again Shell (Bash),9.8
4,CVE-2014-7169,GNU,Bourne-Again Shell (Bash),9.8
5,CVE-2016-10033,PHP,PHPMailer,9.8
6,CVE-2016-4437,Apache,Shiro,9.8
7,CVE-2017-5638,Apache,Struts,9.8
8,CVE-2018-13379,Fortinet,FortiOS,9.8
9,CVE-2018-15961,Adobe,ColdFusion,9.8


## 5 · Interoperability — merge with a file from somebody else
Another team keeps its own notes about CVEs. Their file has **its own vocabulary** (here invented for the demo) but uses the **same `@id`** — the NVD URL.
No join, no mapping table: the two graphs meet at the identifier.

In [9]:
their_file = OUT / "other_team_notes.jsonld"                         # made-up example data, NOT from the project
their_file.write_text(json.dumps({
  "@context": {"ex": "http://example.org/soc#", "priority": "ex:patchPriority", "owner": "ex:ownedBy"},
  "@id": "https://nvd.nist.gov/vuln/detail/CVE-2024-34102",          # same identifier as ours
  "priority": "P1 - patch this week", "owner": "web-platform team"}, indent=1))

g2 = Graph().parse(their_file, format="json-ld")
merged = g + g2                                                      # merging graphs = just adding triples
print(len(g), "+", len(g2), "->", len(merged), "triples")
res = merged.query(P + '''PREFIX ex: <http://example.org/soc#> SELECT ?cve ?product ?score ?priority ?owner WHERE {
  ?v dct:identifier ?cve ; tg:product ?product ; tg:nvdRecord/tg:cvssV31/tg:baseScore ?score ; ex:patchPriority ?priority ; ex:ownedBy ?owner }''')
display(pd.DataFrame([[str(x) for x in r] for r in res], columns=[str(v) for v in res.vars]))
print("One row: CVSS and product come from OUR file, priority and owner from THEIRS, joined only by the @id.")

1944 + 2 -> 1946 triples


,cve,product,score,priority,owner
0,CVE-2024-34102,Commerce and Magento Open Source,9.8,P1 - patch this week,web-platform team


One row: CVSS and product come from OUR file, priority and owner from THEIRS, joined only by the @id.


## What you just saw
* **Export** = a normal API call that returns `application/ld+json`; the file is self-contained (the `@context` travels with it).
* **Reload** needs no knowledge of our code: any JSON-LD library gives triples.
* **Use** = standard SPARQL queries over the triples.
* **Interoperability** = two independent files about the same thing merge because they share the **identifier** (`@id`), not because someone wrote a conversion.

Limits: the `tg:` vocabulary is our own and a placeholder namespace; another team must read our context to understand `tg:` terms. The "other team" file here is invented to show the mechanism.